# Hunting for Regulatory Motifs

In [2]:
import os

def get_dataset_lines(filename):
    path_to_download_folder = os.path.join(os.path.expanduser('~'), 'Downloads', filename)
    with open(path_to_download_folder) as f:
        return f.read().strip().split('\n')

**Exercise Break**: What is the expected number of occurrences of a 9-mer in 500 random DNA strings, each of length 1000? Assume that the sequences are formed by selecting each nucleotide (A, C, G, T) with the same probability (0.25).

In [3]:
l = 1000
n = 500
k = 9

# Probability of a specific k-mer appearing at any given position
prob = 0.25 ** k

# Number of valid starting positions for a k-mer in a string of length l
num_positions = l - k + 1

# Expected number of occurrences in one string
expected_per_string = num_positions * prob

# Total expected occurrences in n strings
total_expected = n * expected_per_string

print(total_expected)

1.89208984375


# Motif Enumeration Problem
Find all (k, d)-motifs in a collection of strings.

**Input**: Integers *k* and *d*, followed by a collection of strings *Dna*.

**Output**: All (*k*, *d*)-motifs in *Dna*.

**Code Challenge**: Implement MotifEnumeration.

**Sample Input**:

```
3 1
ATTTGGC TGCCTTA CGGTATC GAAAATT
```

**Sample Output**:

```
ATA ATT GTT TTT
```

In [4]:
def HammingDistance(p, q):
    count = 0
    for i in range(len(p)):
        if p[i] != q[i]:
            count += 1
    return count

def Neighbors(Pattern, d):
    if d == 0:
        return {Pattern}
    if len(Pattern) == 1:
        return {'A', 'C', 'G', 'T'}
    neighborhood = set()
    SuffixNeighbors = Neighbors(Pattern[1:], d)
    for Text in SuffixNeighbors:
        if HammingDistance(Pattern[1:], Text) < d:
            for x in ['A', 'C', 'G', 'T']:
                neighborhood.add(x + Text)
        else:
            neighborhood.add(Pattern[0] + Text)
    return neighborhood

In [5]:
def MotifEnumeration(Dna, k, d):
    Patterns = set()
    first_seq = Dna[0]
    candidates = set()
    # Generate neighbors for each k-mer in the first string
    for i in range(len(first_seq) - k + 1):
        pattern = first_seq[i:i+k]
        candidates.update(Neighbors(pattern, d))
    
    # Check if each candidate appears in all other strings with at most d mismatches
    for motif in candidates:
        in_all = True
        for seq in Dna:
            found_in_seq = False
            for i in range(len(seq) - k + 1):
                kmer = seq[i:i+k]
                if HammingDistance(motif, kmer) <= d:
                    found_in_seq = True
                    break
            if not found_in_seq:
                in_all = False
                break
        if in_all:
            Patterns.add(motif)
            
    return sorted(list(Patterns))

# Sample Input
k = 3
d = 1
Dna = ["ATTTGGC", "TGCCTTA", "CGGTATC", "GAAAATT"]

motifs = MotifEnumeration(Dna, k, d)
print(*motifs)

# Test Assertion
expected_output = ['ATA', 'ATT', 'GTT', 'TTT']
assert motifs == expected_output, f"Expected {expected_output}, but got {motifs}"
print("Test passed!")

ATA ATT GTT TTT
Test passed!


In [6]:
# Test Dataset
test_dataset_filename = 'dataset_30302_8.txt' 
try:
    lines = get_dataset_lines(test_dataset_filename)
    k, d = map(int, lines[0].split())
    Dna = lines[1].split() 
    
    motifs = MotifEnumeration(Dna, k, d)
    print(*motifs)
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

AAAAA AAAAC AAAAG AAAAT AAACA AAACC AAACG AAACT AAAGA AAAGC AAAGT AAATA AAATC AAATG AACAA AACAC AACAG AACAT AACCA AACCC AACCG AACCT AACGA AACGC AACGG AACGT AACTA AACTC AACTG AACTT AAGAA AAGAC AAGAG AAGAT AAGCA AAGCC AAGCG AAGGA AAGGC AAGGT AAGTA AAGTC AAGTT AATAA AATAC AATAG AATAT AATCA AATCC AATCG AATCT AATGA AATGC AATGG AATGT AATTT ACAAA ACAAC ACAAG ACAAT ACACA ACACC ACACG ACACT ACAGA ACAGC ACAGG ACAGT ACATA ACATC ACATG ACATT ACCAA ACCAC ACCAG ACCAT ACCCA ACCCC ACCGA ACCGC ACCGG ACCGT ACCTA ACCTC ACCTG ACCTT ACGAA ACGAC ACGAG ACGAT ACGCA ACGCC ACGGA ACGGC ACGGT ACGTA ACGTG ACTAA ACTAC ACTAG ACTAT ACTCA ACTGA ACTGC ACTGG ACTTA ACTTG ACTTT AGAAA AGAAC AGAAG AGAAT AGACA AGACC AGACG AGACT AGAGA AGAGC AGAGG AGATA AGATC AGATG AGATT AGCAA AGCAC AGCAG AGCAT AGCCA AGCCG AGCCT AGCGA AGCGC AGCGG AGCGT AGCTG AGCTT AGGAC AGGCA AGGCC AGGCG AGGGA AGGGT AGGTC AGGTG AGTAA AGTCA AGTCC AGTGA AGTGC AGTGG AGTTG ATAAA ATAAC ATAAT ATACA ATACC ATACG ATAGA ATAGC ATAGG ATAGT ATATA ATATC ATATG ATCAC ATCAG ATCC

**Exercise Break**: The minimium possible value of Score(Motifs) is 0 (if all the k-mers in Motifs are the same). What is the maximum possible value of Score(Motifs) for 10 motifs of length 15?

In [7]:
import math

t = 10 # number of motifs
k = 15 # length of motifs

# To maximize the score, we minimize the count of the most frequent nucleotide in each column.
# We distribute t nucleotides as evenly as possible among A, C, G, T.
# ceil(10/4) = 3.
min_consensus_count = math.ceil(t / 4)

max_score_per_column = t - min_consensus_count
max_total_score = max_score_per_column * k

print(f"Maximum possible Score(Motifs): {max_total_score}")

Maximum possible Score(Motifs): 105


## Entropy and the motif logo

**Exercise Break**: Compute the entropy of the NF-κB motif matrix (reproduced below). Your answer should be accurate to within 0.002 of the correct response.

Motifs:

$$
\begin{matrix}
T & C & G & G & G & G & g & T & T & T & t & t \\
c & C & G & G & t & G & A & c & T & T & a & C \\
a & C & G & G & G & G & A & T & T & T & t & C \\
T & t & G & G & G & G & A & c & T & T & t & t \\
a & a & G & G & G & G & A & c & T & T & C & C \\
T & t & G & G & G & G & A & c & T & T & C & C \\
T & C & G & G & G & G & A & T & T & c & a & t \\
T & C & G & G & G & G & A & T & T & c & C & t \\
T & a & G & G & G & G & A & a & c & T & a & C \\
T & C & G & G & G & t & A & T & a & a & C & C \\
\end{matrix}
$$

In [8]:
import math
from collections import Counter

def calculate_entropy(motifs):
    if not motifs:
        return 0
    k = len(motifs[0])
    t = len(motifs)
    entropy = 0
    
    for j in range(k):
        column = [motif[j] for motif in motifs]
        counts = Counter(column)
        col_entropy = 0
        for base in ['A', 'C', 'G', 'T']:
            prob = counts[base] / t
            if prob > 0:
                col_entropy += prob * math.log2(prob)
        entropy -= col_entropy
        
    return entropy

motifs = [
    "TCGGGGGTTTTT",
    "CCGGTGACTTAC",
    "ACGGGGATTTTC",
    "TTGGGGACTTTT",
    "AAGGGGACTTCC",
    "TTGGGGACTTCC",
    "TCGGGGATTCAT",
    "TCGGGGATTCCT",
    "TAGGGGAACTAC",
    "TCGGGTATAACC",
]

entropy = calculate_entropy(motifs)
print(f"Entropy: {entropy}")

Entropy: 9.916290005356972


# Distance Between Pattern and Strings Problem
Compute the distance between a pattern and a collection of strings.

**Input**: A string *Pattern* and a collection of strings *Dna*.

**Output**: *d*(*Pattern*, *Dna*).

**Code Challenge**: Implement DistanceBetweenPatternAndStrings.

**Sample Input**:

```
AAA
TTACCTTAAC GATATCTGTC ACGGCGTTCG CCCTAAAGAG CGTCAGAGGT
```

**Sample Output**:

```
5
```

In [9]:
def DistanceBetweenPatternAndStrings(Pattern, Dna):
    k = len(Pattern)
    distance = 0
    for Text in Dna:
        hamming_distance = float('inf')
        for i in range(len(Text) - k + 1):
            Pattern_prime = Text[i:i+k]
            d = HammingDistance(Pattern, Pattern_prime)
            if d < hamming_distance:
                hamming_distance = d
        distance += hamming_distance
    return distance

# Sample Input
Pattern = "AAA"
Dna = ["TTACCTTAAC", "GATATCTGTC", "ACGGCGTTCG", "CCCTAAAGAG", "CGTCAGAGGT"]
distance = DistanceBetweenPatternAndStrings(Pattern, Dna)
print(distance)

# Test Assertion
expected_output = 5
assert distance == expected_output, f"Expected {expected_output}, but got {distance}"
print("Test passed!")

5
Test passed!


In [10]:
# Test Dataset
test_dataset_filename = 'dataset_30312_1.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    Pattern = lines[0]
    Dna = lines[1].split()
    
    distance = DistanceBetweenPatternAndStrings(Pattern, Dna)
    print(distance)
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

71


# Motif Finding Problem
Given a collection of strings, find a set of *k*-mers, one from each string, that minimizes the score of the resulting motif.

**Input**: A collection of strings *Dna* and an integer *k*.

**Output**: A collection *Motifs* of *k*-mers, one from each string in *Dna*, minimizing *Score*(*Motifs*) among all possible choices of *k*-mers.

# Median String Problem
Find a median string.

**Input**: An integer *k* and a collection of strings *Dna*.

**Output**: A *k*-mer *Pattern* that minimizes *d*(*Pattern*, *Dna*) among all possible choices of *k*-mers.

**Code Challenge**: Implement MedianString.

**Sample Input**:

```
3
AAATTGACGCAT GACGACCACGTT CGTCAGCGCCTG GCTGAGCACCGG AGTTCGGGACAG
```

**Sample Output**:

```
GAC
```

In [11]:
import itertools

def AllStrings(k):
    return ["".join(p) for p in itertools.product('ACGT', repeat=k)]

def MedianString(Dna, k):
    distance = float('inf')
    Median = ""
    Patterns = AllStrings(k)
    for Pattern in Patterns:
        d = DistanceBetweenPatternAndStrings(Pattern, Dna)
        if distance > d:
            distance = d
            Median = Pattern
    return Median

# Sample Input
k = 3
Dna = ["AAATTGACGCAT", "GACGACCACGTT", "CGTCAGCGCCTG", "GCTGAGCACCGG", "AGTTCGGGACAG"]
median = MedianString(Dna, k)
print(median)

# Test Assertion
expected_output = 'GAC'
assert median == expected_output, f"Expected {expected_output}, but got {median}"
print("Test passed!")

GAC
Test passed!


In [12]:
# Test Dataset
test_dataset_filename = 'dataset_30304_9.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    k = int(lines[0])
    Dna = lines[1].split()
    
    median = MedianString(Dna, k)
    print(median)
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

AAGACA


# Profile-most Probable k-mer Problem
Find a Profile-most probable *k*-mer in a string.

**Input**: A string *Text*, an integer *k*, and a 4 × *k* matrix *Profile*.

**Output**: A Profile-most probable *k*-mer in *Text*.

**Code Challenge**: Solve the Profile-most Probable k-mer Problem.

**Sample Input**:

```
ACCTGTTTATTGCCTAAGTTCCGAACAAACCCAATATAGCCCGAGGGCCT
5
0.2 0.2 0.3 0.2 0.3
0.4 0.3 0.1 0.5 0.1
0.3 0.3 0.5 0.2 0.4
0.1 0.2 0.1 0.1 0.2
```

**Sample Output**:

```
CCGAG
```

In [13]:
def Pr(Pattern, Profile):
    prob = 1
    for i in range(len(Pattern)):
        prob *= Profile[Pattern[i]][i]
    return prob

def ProfileMostProbableKmer(Text, k, Profile):
    max_prob = -1
    most_probable = Text[0:k]
    
    for i in range(len(Text) - k + 1):
        Pattern = Text[i:i+k]
        prob = Pr(Pattern, Profile)
        if prob > max_prob:
            max_prob = prob
            most_probable = Pattern
            
    return most_probable

# Sample Input
Text = "ACCTGTTTATTGCCTAAGTTCCGAACAAACCCAATATAGCCCGAGGGCCT"
k = 5
# Profile matrix represented as a dictionary for easier access
Profile = {
    'A': [0.2, 0.2, 0.3, 0.2, 0.3],
    'C': [0.4, 0.3, 0.1, 0.5, 0.1],
    'G': [0.3, 0.3, 0.5, 0.2, 0.4],
    'T': [0.1, 0.2, 0.1, 0.1, 0.2]
}

result = ProfileMostProbableKmer(Text, k, Profile)
print(result)

# Test Assertion
expected_output = 'CCGAG'
assert result == expected_output, f"Expected {expected_output}, but got {result}"
print("Test passed!")

CCGAG
Test passed!


In [14]:
# Test Dataset
test_dataset_filename = 'dataset_30305_3.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    Text = lines[0]
    k = int(lines[1])
    
    # Parse Profile matrix
    # The input format usually has the matrix rows separated by spaces
    # We need to read the next 4 lines
    profile_lines = lines[2:]
    Profile = {
        'A': list(map(float, profile_lines[0].split())),
        'C': list(map(float, profile_lines[1].split())),
        'G': list(map(float, profile_lines[2].split())),
        'T': list(map(float, profile_lines[3].split()))
    }
    
    result = ProfileMostProbableKmer(Text, k, Profile)
    print(result)
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

GTTAACACGCATGT


# Greedy Motif Search Problem
Implement GreedyMotifSearch.

**Input**: Integers *k* and *t*, followed by a collection of strings *Dna*.

**Output**: A collection of strings *BestMotifs* resulting from applying GreedyMotifSearch(*Dna*, *k*, *t*).

**Code Challenge**: Implement GreedyMotifSearch.

**Sample Input**:

```
3 5
GGCGTTCAGGCA AAGAATCAGTCA CAAGGAGTTCGC CACGTCAATCAC CAATAATATTCG
```

**Sample Output**:

```
CAG CAG CAA CAA CAA
```

In [15]:
def Count(Motifs):
    motif_length = len(Motifs[0])
    number_of_motifs = len(Motifs)
    
    counts = {}
    for nucleotide in "ACGT":
        counts[nucleotide] = [0] * motif_length
            
    for i in range(number_of_motifs):
        for j in range(motif_length):
            nucleotide = Motifs[i][j]
            counts[nucleotide][j] += 1
    return counts

def Score(Motifs):
    counts = Count(Motifs)
    motif_length = len(Motifs[0])
    number_of_motifs = len(Motifs)
    score = 0
    for j in range(motif_length):
        max_count = max(counts[base][j] for base in 'ACGT')
        score += (number_of_motifs - max_count)
    return score

def FormProfile(Motifs):
    counts = Count(Motifs)
    number_of_motifs = len(Motifs)
    profile = {}
    for base in 'ACGT':
        profile[base] = [count / number_of_motifs for count in counts[base]]
    return profile

def GreedyMotifSearch(Dna, k, t):
    BestMotifs = [text[0:k] for text in Dna]
    
    dna_string_length = len(Dna[0])
    for i in range(dna_string_length - k + 1):
        Motifs = []
        Motifs.append(Dna[0][i:i+k])
        for j in range(1, t):
            Profile = FormProfile(Motifs)
            Motifs.append(ProfileMostProbableKmer(Dna[j], k, Profile))
        
        if Score(Motifs) < Score(BestMotifs):
            BestMotifs = Motifs
            
    return BestMotifs

# Sample Input
k = 3
t = 5
Dna = ["GGCGTTCAGGCA", "AAGAATCAGTCA", "CAAGGAGTTCGC", "CACGTCAATCAC", "CAATAATATTCG"]

best_motifs = GreedyMotifSearch(Dna, k, t)
print(*best_motifs)

# Test Assertion
expected_output = ['CAG', 'CAG', 'CAA', 'CAA', 'CAA']
assert best_motifs == expected_output, f"Expected {expected_output}, but got {best_motifs}"
print("Test passed!")

CAG CAG CAA CAA CAA
Test passed!


In [16]:
# Test Dataset
test_dataset_filename = 'dataset_30305_5.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    k, t = map(int, lines[0].split())
    Dna = lines[1].split()
    
    best_motifs = GreedyMotifSearch(Dna, k, t)
    print(*best_motifs)
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

TACTGTAGTCTA GCCGAGGATTTT GGCCCGCCTGTT AGCTTAGCTCGC CTTCTTGTCCTG AATTTGGGTTTT CATTGGGCTCTT TGTTGGGGTGTT GGTTGGGTTGTT TCTTGGGATCTT GCTTGGGGTTTT TGTTTTGACGTT GTTTGGGGTCTT TGTTGGGGTGTT CTTTGGGATTTT CATTGGGATTTT CGTTGGGGTCTT TCTTGGGATCTT TATTGGGATCTT ATTTGGGTTCTT GGTTGGGATTTT CATTGGGGTCGA GTTTGGGGTCTT GTTTGGGGTCTT GTTTTTGGTGTG


# Greedy Motif Search with Pseudocounts Problem
Implement GreedyMotifSearch with pseudocounts.

**Input**: Integers *k* and *t*, followed by a collection of strings *Dna*.

**Output**: A collection of strings *BestMotifs* resulting from applying GreedyMotifSearch(*Dna*, *k*, *t*) with pseudocounts.

**Code Challenge**: Implement GreedyMotifSearch with pseudocounts.

**Sample Input**:

```
3 5
GGCGTTCAGGCA AAGAATCAGTCA CAAGGAGTTCGC CACGTCAATCAC CAATAATATTCG
```

**Sample Output**:

```
TTC ATC TTC ATC TTC
```

In [17]:
def CountWithPseudocounts(Motifs):
    motif_length = len(Motifs[0])
    number_of_motifs = len(Motifs)
    
    counts = {}
    for nucleotide in "ACGT":
        counts[nucleotide] = [1] * motif_length
            
    for i in range(number_of_motifs):
        for j in range(motif_length):
            nucleotide = Motifs[i][j]
            counts[nucleotide][j] += 1
    return counts

def FormProfileWithPseudocounts(Motifs):
    counts = CountWithPseudocounts(Motifs)
    number_of_motifs = len(Motifs)
    profile = {}
    for base in 'ACGT':
        profile[base] = [count / (number_of_motifs + 4) for count in counts[base]]
    return profile

def GreedyMotifSearchWithPseudocounts(Dna, k, t):
    BestMotifs = [text[0:k] for text in Dna]
    
    dna_string_length = len(Dna[0])
    for i in range(dna_string_length - k + 1):
        Motifs = []
        Motifs.append(Dna[0][i:i+k])
        for j in range(1, t):
            Profile = FormProfileWithPseudocounts(Motifs)
            Motifs.append(ProfileMostProbableKmer(Dna[j], k, Profile))
        
        if Score(Motifs) < Score(BestMotifs):
            BestMotifs = Motifs
            
    return BestMotifs

# Sample Input
k = 3
t = 5
Dna = ["GGCGTTCAGGCA", "AAGAATCAGTCA", "CAAGGAGTTCGC", "CACGTCAATCAC", "CAATAATATTCG"]

best_motifs = GreedyMotifSearchWithPseudocounts(Dna, k, t)
print(*best_motifs)

# Test Assertion
expected_output = ['TTC', 'ATC', 'TTC', 'ATC', 'TTC']
assert best_motifs == expected_output, f"Expected {expected_output}, but got {best_motifs}"
print("Test passed!")

TTC ATC TTC ATC TTC
Test passed!


In [18]:
# Test Dataset
test_dataset_filename = 'dataset_30306_9.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    k, t = map(int, lines[0].split())
    Dna = lines[1].split()
    
    best_motifs = GreedyMotifSearchWithPseudocounts(Dna, k, t)
    print(*best_motifs)
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

GGTCGTCCTAAG GGTCACCTTAAG GGTCATCATAAG TGTCACCCTAAG TGTCGCCTTAAG TGTCGCCTTAAG GGTCGGCTTAAG CGTCAACTTAAG GGTCACCGTAAG GGTCCGCCTAAG GGTCAACATAAG GGTCCCCGTAAG GGTCGACTTAAG TGTCGACTTAAG AGTCGGCATAAG CGTCATCATAAG GGTCTACGTAAG GGTCATCATAAG TGTCGGCGTAAG TGTCTACATAAG GCTCTCCCTAAC CGTCTGCTTAAG AGTCGCCTTAAG TGTCTCCGTAAG CGTCCACGTAAG


In [23]:
# Corusera Quiz Questions

# Q5

def AllMedianStrings(Dna, k):
    distance = float('inf')
    Medians = []
    Patterns = AllStrings(k)
    for Pattern in Patterns:
        d = DistanceBetweenPatternAndStrings(Pattern, Dna)
        if distance > d:
            distance = d
            Medians = [Pattern]
        elif distance == d:
            Medians.append(Pattern)
    return Medians

Dna = [
    "CTCGATGAGTAGGAAAGTAGTTTCACTGGGCGAACCACCCCGGCGCTAATCCTAGTGCCC",
    "GCAATCCTACCCGAGGCCACATATCAGTAGGAACTAGAACCACCACGGGTGGCTAGTTTC",
    "GGTGTTGAACCACGGGGTTAGTTTCATCTATTGTAGGAATCGGCTTCAAATCCTACACAG"
]
k = 7

medians = AllMedianStrings(Dna, k)
print("All Median Strings:", medians)


# Q6
Profile = {
    'A': [0.4, 0.3, 0.0, 0.1, 0.0, 0.9],
    'C': [0.2, 0.3, 0.0, 0.4, 0.0, 0.1],
    'G': [0.1, 0.3, 1.0, 0.1, 0.5, 0.0],
    'T': [0.3, 0.1, 0.0, 0.4, 0.5, 0.0]
}
Pattern = "AAGTTC"
print(f"Probability of pattern {Pattern}: {Pr(Pattern, Profile)}")

All Median Strings: ['AATCCTA', 'GAACCAC', 'GTAGGAA', 'TAGTTTC']
Probability of pattern AAGTTC: 0.0024000000000000002
